# 03. Inference

Predict new text with a trained model, and check that merging LoRA into the base weights does not change the results.

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Change this to your repo URL. For a private repo, add a GITHUB_TOKEN secret on Colab.
REPO_URL = "https://github.com/YOUR-USERNAME/vihsd-lora.git"

def github_token():
    if os.environ.get("GITHUB_TOKEN"):
        return os.environ["GITHUB_TOKEN"]
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None

def find_repo():
    places = [Path.cwd(), *Path.cwd().parents, Path("/content/vihsd-lora"), Path("/kaggle/working/vihsd-lora")]
    places += list(Path("/kaggle/input").glob("*")) + list(Path("/kaggle/input").glob("*/vihsd-lora"))
    return next((place for place in places if (place / "src" / "vihsd_lora").is_dir()), None)

repo = find_repo()
if repo is None:
    if "YOUR-USERNAME" in REPO_URL:
        raise ValueError("Set REPO_URL to your GitHub repo URL, then run this cell again.")
    repo = Path("/content" if Path("/content").exists() else "/kaggle/working") / "vihsd-lora"
    token = github_token()
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", url, str(repo)], check=True)
elif (repo / ".git").exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=False)
if not os.environ.get("VIHSD_SKIP_INSTALL"):
    # Colab ships an old torchao that makes peft fail when attaching LoRA. This repo does not use torchao, so remove it.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
sys.path.insert(0, str(repo / "src"))
print("Repo:", repo)

In [ ]:
from vihsd_lora.config import Settings
from vihsd_lora.env import default_output_root, describe_gpu, detect_platform, free_space_gb, load_secrets

print("Platform:", detect_platform())
print("GPU:", describe_gpu())
print("Secrets found:", load_secrets())

settings = Settings.from_env(output_root=default_output_root())
free_gb = free_space_gb(settings.output_root)
print("Results are saved to:", settings.output_root)
print(f"Free space: {free_gb} GB")
if free_gb < 6:
    print("Warning: less than 6GB left. A full fine-tune run needs about 3GB, so clean up Drive and empty the trash.")

In [ ]:
from pathlib import Path
import pandas as pd
from vihsd_lora.evaluate import select_for_test
from vihsd_lora.inference import Predictor

SLUG = select_for_test(settings)["lora"]
predictor = Predictor(Path(settings.output_root) / "runs" / SLUG)
print("Using run:", SLUG)

## Predict

Replace the list below with the Vietnamese sentences you want to try.

In [ ]:
TEXTS = [
    "Hôm nay thời tiết đẹp quá, cả nhà đi chơi vui nhé.",  # "The weather is so nice today, have fun everyone."
    "Bài viết rất hữu ích, cảm ơn bạn đã chia sẻ.",  # "Very useful post, thanks for sharing."
]
before = pd.DataFrame(predictor.predict(TEXTS))
display(before)

## Merge LoRA into the base weights

After merging, the results should stay almost the same. This is why LoRA adds no latency at deployment.

In [ ]:
latency_adapter = predictor.latency_ms(TEXTS, batch_size=1)
predictor.merge_lora()
latency_merged = predictor.latency_ms(TEXTS, batch_size=1)
after = pd.DataFrame(predictor.predict(TEXTS))

print(f"Latency per sentence: unmerged {latency_adapter} ms, merged {latency_merged} ms")
print("Labels unchanged:", (before["label"] == after["label"]).all())
display(after)